In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

!pip uninstall -y torchao torchaudio diffusers -q
!pip install -U bitsandbytes -q
!pip install git+https://github.com/huggingface/diffusers.git -q
!pip install -U transformers accelerate -q

print("✅ تم تثبيت المكتبات")

In [1]:
import torch
import diffusers
import bitsandbytes as bnb

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"diffusers: {diffusers.__version__}")
print(f"QwenImage21Pipeline exists: {hasattr(diffusers, 'QwenImage21Pipeline')}")
print(f"bitsandbytes: {bnb.__version__}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB
diffusers: 0.41.0.dev0
QwenImage21Pipeline exists: True
bitsandbytes: 0.50.2


In [2]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import requests
import json

# تسجيل الدخول لـ Hugging Face
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("✅ تم تسجيل الدخول إلى Hugging Face")

# فحص بنية النموذج بدون تحميله
url = "https://huggingface.co/Qwen/Qwen-Image-2.1/raw/main/model_index.json"
model_index = requests.get(url, timeout=30).json()

print("\n📄 model_index.json:")
print(json.dumps(model_index, indent=2))

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
✅ تم تسجيل الدخول إلى Hugging Face

📄 model_index.json:
{
  "_class_name": "QwenImage21Pipeline",
  "_diffusers_version": "0.37.0.dev0",
  "processor": [
    "transformers",
    "Qwen3VLProcessor"
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "Qwen3VLForConditionalGeneration"
  ],
  "transformer": [
    "diffusers",
    "QwenImage21Transformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLQwenImage21"
  ]
}


In [3]:
import torch

from diffusers import (
    QwenImage21Transformer2DModel,
    AutoencoderKLQwenImage21,
    BitsAndBytesConfig as DiffusersBnBConfig,
)

from transformers import (
    Qwen3VLForConditionalGeneration,
    Qwen3VLProcessor,
    BitsAndBytesConfig as TransformersBnBConfig,
)

print("✅ QwenImage21Transformer2DModel imported")
print("✅ AutoencoderKLQwenImage21 imported")
print("✅ Qwen3VLForConditionalGeneration imported")
print("✅ Qwen3VLProcessor imported")
print("✅ BitsAndBytesConfig imported from diffusers and transformers")

✅ QwenImage21Transformer2DModel imported
✅ AutoencoderKLQwenImage21 imported
✅ Qwen3VLForConditionalGeneration imported
✅ Qwen3VLProcessor imported
✅ BitsAndBytesConfig imported from diffusers and transformers


In [4]:
import torch
import gc

from diffusers import (
    QwenImage21Transformer2DModel,
    BitsAndBytesConfig as DiffusersBnBConfig,
)

# تنظيف الذاكرة قبل التحميل
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen-Image-2.1"

print("⏳ جاري تحميل الـ transformer بنسخة 4-bit...")

transformer_quant_config = DiffusersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

transformer = QwenImage21Transformer2DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    quantization_config=transformer_quant_config,
    dtype=torch.float16,
)

print("✅ تم تحميل الـ transformer")

print("\n📊 حالة ذاكرة GPU بعد تحميل الـ transformer:")
print(f"   allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

⏳ جاري تحميل الـ transformer بنسخة 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

(…)ion_pytorch_model.safetensors.index.json:   0%|          | 0.00/30.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ تم تحميل الـ transformer

📊 حالة ذاكرة GPU بعد تحميل الـ transformer:
   allocated: 4.00 GB
   reserved:  4.10 GB


In [5]:
import torch
import gc

from transformers import (
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig as TransformersBnBConfig,
)

MODEL_ID = "Qwen/Qwen-Image-2.1"

print("📊 ذاكرة GPU قبل تحميل الـ text encoder:")
print(f"   allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

gc.collect()
torch.cuda.empty_cache()

print("\n⏳ جاري تحميل الـ text encoder بنسخة 4-bit...")

text_encoder_quant_config = TransformersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

text_encoder = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    quantization_config=text_encoder_quant_config,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("✅ تم تحميل الـ text encoder")

print("\n📊 ذاكرة GPU بعد تحميل الـ text encoder:")
print(f"   allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

📊 ذاكرة GPU قبل تحميل الـ text encoder:
   allocated: 4.00 GB
   reserved:  4.10 GB

⏳ جاري تحميل الـ text encoder بنسخة 4-bit...


config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

✅ تم تحميل الـ text encoder

📊 ذاكرة GPU بعد تحميل الـ text encoder:
   allocated: 10.74 GB
   reserved:  10.83 GB


In [6]:
import torch
import gc
import inspect

from diffusers import AutoencoderKLQwenImage21, QwenImage21Pipeline

MODEL_ID = "Qwen/Qwen-Image-2.1"

print("📊 ذاكرة GPU قبل تحميل الـ VAE:")
print(f"   allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

gc.collect()
torch.cuda.empty_cache()

print("\n⏳ جاري تحميل الـ VAE...")

vae = AutoencoderKLQwenImage21.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    dtype=torch.float16,
)

vae = vae.to("cuda")

print("✅ تم تحميل الـ VAE")

print("\n📊 ذاكرة GPU بعد تحميل الـ VAE:")
print(f"   allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

print("\n🔍 توقيع دالة بناء QwenImage21Pipeline:")
print(inspect.signature(QwenImage21Pipeline.__init__))

📊 ذاكرة GPU قبل تحميل الـ VAE:
   allocated: 10.74 GB
   reserved:  10.83 GB

⏳ جاري تحميل الـ VAE...


config.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.35GB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

✅ تم تحميل الـ VAE

📊 ذاكرة GPU بعد تحميل الـ VAE:
   allocated: 11.42 GB
   reserved:  11.46 GB

🔍 توقيع دالة بناء QwenImage21Pipeline:
(self, scheduler: diffusers.schedulers.scheduling_flow_match_euler_discrete.FlowMatchEulerDiscreteScheduler, vae: diffusers.models.autoencoders.autoencoder_kl_qwenimage21.AutoencoderKLQwenImage21, text_encoder: transformers.models.qwen3_vl.modeling_qwen3_vl.Qwen3VLForConditionalGeneration, processor: transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor, transformer: diffusers.models.transformers.transformer_qwenimage21.QwenImage21Transformer2DModel)
